# Figure 1b-d codes

This notebook contains the plotting code for main-text Figure 1b, Figure 1c, and Figure 1d using `data/figure1_node_state_proportions.csv`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'data' / 'figure1_node_state_proportions.csv'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25

COLORS = sns.color_palette('vlag', 5)
SHORT_COLORS = [COLORS[0], COLORS[2], COLORS[4]]
TOPIC_ORDER = ['Politics', 'Gun_Control', 'Abortion']
TITLE_BY_TOPIC = {
    'Politics': 'Partisan Alignment',
    'Gun_Control': 'Gun Control',
    'Abortion': 'Abortion Ban',
}
STATE_ORDER = [2, 1, 0, -1, -2]


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
def state_matrix(topic):
    topic_df = df[df['topic'] == topic]
    matrix = (
        topic_df.pivot(index='state_order', columns='epoch', values='proportion')
        .sort_index()
        .to_numpy()
    )
    return matrix

for topic in TOPIC_ORDER:
    matrix = state_matrix(topic)
    assert matrix.shape == (5, 41)
    np.testing.assert_allclose(matrix.sum(axis=0), 1.0)


## Figure 1b


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for i, topic in enumerate(TOPIC_ORDER):
    data_matrix = np.cumsum(state_matrix(topic), axis=0)
    data_matrix = np.concatenate([np.zeros((1, data_matrix.shape[1])), data_matrix], axis=0)

    for state_idx in range(1, 6):
        line_pre = data_matrix[state_idx - 1, :]
        line_after = data_matrix[state_idx, :]
        axes[i].fill_between(np.arange(line_pre.shape[0]), line_pre, line_after, color=COLORS[state_idx - 1])

    axes[i].set_ylim(0, 1)
    axes[i].set_xlim(0, line_pre.shape[0] - 1)
    axes[i].set_xticks(range(0, line_pre.shape[0], 10))
    axes[i].set_xlabel('Time')
    axes[i].set_title(TITLE_BY_TOPIC[topic], fontsize=25, fontweight='bold')
    if i == 0:
        axes[i].set_ylabel('Proportion')

fig.savefig(OUTPUT_DIR / 'figure1b_opinion_dynamics.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()


## Figure 1c


In [ ]:
for topic in TOPIC_ORDER:
    data_matrix = state_matrix(topic)
    title = TITLE_BY_TOPIC[topic]

    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.bar(np.arange(5), data_matrix[:, 0], color=COLORS, edgecolor='k')
    ax.set_xticks([])
    ax.set_title('Initialization', fontsize=25, fontweight='bold')
    ax.set_ylabel('Proportion')
    ax.set_yticks([0, 0.2, 0.4])
    fig.savefig(OUTPUT_DIR / f"figure1c_{title.replace(' ', '_').lower()}_init.pdf", bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.bar(np.arange(5), data_matrix[:, -1], color=COLORS, edgecolor='k')
    ax.set_xticks([])
    ax.set_title(title, fontsize=25, fontweight='bold')
    if topic == 'Politics':
        ax.set_ylabel('Proportion')
    fig.savefig(OUTPUT_DIR / f"figure1c_{title.replace(' ', '_').lower()}_final.pdf", bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()


## Figure 1d


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 4))

for i, topic in enumerate(TOPIC_ORDER):
    final_state = state_matrix(topic)[:, -1]
    sizes = [
        final_state[0] + final_state[1],
        final_state[2],
        final_state[3] + final_state[4],
    ]
    wedges = axs[i].pie(sizes, colors=SHORT_COLORS, startangle=90)[0]
    for wedge in wedges:
        wedge.set_edgecolor('k')
        wedge.set_linewidth(1)
    axs[i].set_title(TITLE_BY_TOPIC[topic], fontsize=25, fontweight='bold')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure1d_opinion_pie.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()
